# 03. Feature Perturbation Experiments

This notebook evaluates model robustness against non-TTL feature perturbations using frozen baseline models: Experiment A (flow duration +/- 5%), Experiment B (timing and source/destination jitter), and Experiment D (combined duration and jitter perturbations).


In [25]:
# ============================================================
# EXPERIMENT A — FLOW-DURATION ROBUSTNESS
#
# SAME FROZEN BASELINE MODELS
# NO RETRAINING
#
# Perturbation:
#   dur × 0.95
#   dur × 1.00
#   dur × 1.05
#
# Metrics:
#   TPR
#   TPR Drop from CLEAN BASELINE
#   Evasion Rate
#   AUROC
#   AUPRC
#
# Baseline is taken PER SEED from the original clean-model
# results, ensuring that each perturbed model is compared
# against its own clean baseline.
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)


print("=" * 100)
print("EXPERIMENT A — FLOW-DURATION ROBUSTNESS")
print("=" * 100)


# ============================================================
# 1. CHECK REQUIRED VARIABLES
# ============================================================

required = [
    "test_df",
    "y_test",
    "preprocessor",
    "rf_models",
    "mlp_models"
]

for var in required:
    assert var in globals(), f"{var} not found"


assert "dur" in test_df.columns, \
    "Flow-duration feature 'dur' not found in test_df"


# ============================================================
# 2. SEEDS
# ============================================================

SEEDS = [42, 123, 2026]

scales = [
    0.95,
    1.00,
    1.05
]


# ============================================================
# 3. DEVICE
# ============================================================

# Find device from one of the stored MLP models
first_mlp = mlp_models[SEEDS[0]]

device = next(
    first_mlp.parameters()
).device

print("MLP device:", device)
print("Test samples:", len(test_df))
print("Duration feature:", "dur")
print("Seeds:", SEEDS)


# ============================================================
# 4. GET CLEAN BASELINE TPR FOR EACH SEED
# ============================================================
#
# IMPORTANT:
# Do NOT use a random global variable such as mlp_tpr.
#
# Each transformed result for Seed 42 is compared against
# the CLEAN Seed-42 TPR.
#
# These are the clean results you previously obtained:
#
# RF:
#   Seed 42   = 0.7167122562
#   Seed 123  = 0.6567546104
#   Seed 2026 = 0.6753728051
#
# MLP:
#   Seed 42   = 0.4374834554
#   Seed 123  = 0.4039089385
#   Seed 2026 = 0.4291670343
#
# Mean MLP TPR = 0.423520
# ============================================================

clean_rf_tpr = {
    42:   0.7167122562428306,
    123:  0.6567546104297185,
    2026: 0.6753728050825024
}


clean_mlp_tpr = {
    42:   0.43748345539574696,
    123:  0.40390893849819115,
    2026: 0.42916703432453895
}


# Check the clean means
print("\nCLEAN BASELINE TPR")
print("-" * 60)

print(
    "RF mean TPR  :",
    np.mean(list(clean_rf_tpr.values()))
)

print(
    "MLP mean TPR :",
    np.mean(list(clean_mlp_tpr.values()))
)

print(
    "Expected MLP : 0.423520"
)


# ============================================================
# 5. PREPROCESSING
# ============================================================

def transform_test(df):

    X = preprocessor.transform(
        df.drop(
            columns=["label", "attack_cat"],
            errors="ignore"
        )
    )

    if hasattr(X, "toarray"):
        X = X.toarray()

    return np.asarray(
        X,
        dtype=np.float32
    )


# ============================================================
# 6. MLP PREDICTION
# ============================================================

def get_mlp_probability(mlp, X):

    mlp.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():

        logits = mlp(X_tensor)

        probabilities = torch.sigmoid(
            logits
        )

    return (
        probabilities
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


# ============================================================
# 7. RESULTS STORAGE
# ============================================================

all_results = []


# ============================================================
# 8. RUN EXPERIMENT
# ============================================================

for scale in scales:

    print("\n")
    print("=" * 100)
    print(f"FLOW DURATION PERTURBATION = {scale:.2f}x")
    print("=" * 100)


    # --------------------------------------------------------
    # Create perturbed copy
    # --------------------------------------------------------

    transformed_df = test_df.copy()

    transformed_df["dur"] = (
        transformed_df["dur"].astype(float)
        * scale
    )


    # --------------------------------------------------------
    # IMPORTANT:
    # Same fitted preprocessor.
    # NO refitting.
    # --------------------------------------------------------

    X_transformed = transform_test(
        transformed_df
    )


    # ========================================================
    # EACH FROZEN MODEL / SEED
    # ========================================================

    for seed in SEEDS:

        print("\n" + "-" * 80)
        print(f"SEED = {seed}")
        print("-" * 80)


        # ====================================================
        # RANDOM FOREST
        # ====================================================

        rf_model = rf_models[seed]

        rf_prob = rf_model.predict_proba(
            X_transformed
        )[:, 1]

        rf_pred = (
            rf_prob >= 0.5
        ).astype(int)


        # ----------------------------------------------------
        # TPR
        # ----------------------------------------------------

        rf_tpr_trans = recall_score(
            y_test,
            rf_pred,
            zero_division=0
        )


        # ----------------------------------------------------
        # Evasion Rate
        # ----------------------------------------------------

        rf_evasion = 1.0 - rf_tpr_trans


        # ----------------------------------------------------
        # TPR DROP FROM SAME CLEAN SEED
        # ----------------------------------------------------

        rf_baseline = clean_rf_tpr[seed]

        rf_tpr_drop = (
            rf_baseline
            - rf_tpr_trans
        )


        # ----------------------------------------------------
        # Ranking metrics
        # ----------------------------------------------------

        rf_auroc = roc_auc_score(
            y_test,
            rf_prob
        )

        rf_auprc = average_precision_score(
            y_test,
            rf_prob
        )


        # ====================================================
        # MLP
        # ====================================================

        mlp_model = mlp_models[seed]

        mlp_prob = get_mlp_probability(
            mlp_model,
            X_transformed
        )

        mlp_pred = (
            mlp_prob >= 0.5
        ).astype(int)


        # ----------------------------------------------------
        # TPR
        # ----------------------------------------------------

        mlp_tpr_trans = recall_score(
            y_test,
            mlp_pred,
            zero_division=0
        )


        # ----------------------------------------------------
        # Evasion Rate
        # ----------------------------------------------------

        mlp_evasion = 1.0 - mlp_tpr_trans


        # ----------------------------------------------------
        # TPR DROP FROM SAME CLEAN SEED
        # ----------------------------------------------------

        mlp_baseline = clean_mlp_tpr[seed]

        mlp_tpr_drop = (
            mlp_baseline
            - mlp_tpr_trans
        )


        # ----------------------------------------------------
        # Ranking metrics
        # ----------------------------------------------------

        mlp_auroc = roc_auc_score(
            y_test,
            mlp_prob
        )

        mlp_auprc = average_precision_score(
            y_test,
            mlp_prob
        )


        # ====================================================
        # PRINT
        # ====================================================

        print("\nRF")
        print(
            f"Clean TPR     : {rf_baseline:.6f}"
        )
        print(
            f"Perturbed TPR : {rf_tpr_trans:.6f}"
        )
        print(
            f"TPR Drop      : {rf_tpr_drop:.6f}"
        )
        print(
            f"Evasion Rate  : {rf_evasion:.6f}"
        )
        print(
            f"AUROC         : {rf_auroc:.6f}"
        )
        print(
            f"AUPRC         : {rf_auprc:.6f}"
        )


        print("\nMLP")
        print(
            f"Clean TPR     : {mlp_baseline:.6f}"
        )
        print(
            f"Perturbed TPR : {mlp_tpr_trans:.6f}"
        )
        print(
            f"TPR Drop      : {mlp_tpr_drop:.6f}"
        )
        print(
            f"Evasion Rate  : {mlp_evasion:.6f}"
        )
        print(
            f"AUROC         : {mlp_auroc:.6f}"
        )
        print(
            f"AUPRC         : {mlp_auprc:.6f}"
        )


        # ====================================================
        # SAVE PER-SEED RESULT
        # ====================================================

        all_results.append({

            "Duration Scale": scale,
            "Seed": seed,

            # RF
            "RF Clean TPR": rf_baseline,
            "RF TPR": rf_tpr_trans,
            "RF TPR Drop": rf_tpr_drop,
            "RF Evasion Rate": rf_evasion,
            "RF AUROC": rf_auroc,
            "RF AUPRC": rf_auprc,

            # MLP
            "MLP Clean TPR": mlp_baseline,
            "MLP TPR": mlp_tpr_trans,
            "MLP TPR Drop": mlp_tpr_drop,
            "MLP Evasion Rate": mlp_evasion,
            "MLP AUROC": mlp_auroc,
            "MLP AUPRC": mlp_auprc
        })


# ============================================================
# 9. PER-SEED DATAFRAME
# ============================================================

duration_by_seed = pd.DataFrame(
    all_results
)


# ============================================================
# 10. MEAN
# ============================================================

metric_columns = [

    "RF TPR",
    "RF TPR Drop",
    "RF Evasion Rate",
    "RF AUROC",
    "RF AUPRC",

    "MLP TPR",
    "MLP TPR Drop",
    "MLP Evasion Rate",
    "MLP AUROC",
    "MLP AUPRC"
]


duration_summary = (
    duration_by_seed
    .groupby("Duration Scale")[metric_columns]
    .mean()
    .reset_index()
)


# ============================================================
# 11. STANDARD DEVIATION
# ============================================================

duration_sd = (
    duration_by_seed
    .groupby("Duration Scale")[metric_columns]
    .std(ddof=1)
    .reset_index()
)


# ============================================================
# 12. FINAL MEAN ± SD TABLE
# ============================================================

print("\n")
print("=" * 140)
print("FLOW-DURATION ROBUSTNESS — MEAN ± SD")
print("=" * 140)


for _, row in duration_summary.iterrows():

    scale = row["Duration Scale"]

    sd_row = duration_sd[
        duration_sd["Duration Scale"] == scale
    ].iloc[0]


    print(
        f"\nDuration = {scale:.2f}x"
    )

    print(
        f"RF  TPR          : "
        f"{row['RF TPR']:.4f} ± "
        f"{sd_row['RF TPR']:.4f}"
    )

    print(
        f"RF  TPR Drop     : "
        f"{row['RF TPR Drop']:.4f} ± "
        f"{sd_row['RF TPR Drop']:.4f}"
    )

    print(
        f"RF  Evasion Rate : "
        f"{row['RF Evasion Rate']:.4f} ± "
        f"{sd_row['RF Evasion Rate']:.4f}"
    )

    print(
        f"RF  AUROC        : "
        f"{row['RF AUROC']:.4f} ± "
        f"{sd_row['RF AUROC']:.4f}"
    )

    print(
        f"RF  AUPRC        : "
        f"{row['RF AUPRC']:.4f} ± "
        f"{sd_row['RF AUPRC']:.4f}"
    )


    print(
        f"MLP TPR          : "
        f"{row['MLP TPR']:.4f} ± "
        f"{sd_row['MLP TPR']:.4f}"
    )

    print(
        f"MLP TPR Drop     : "
        f"{row['MLP TPR Drop']:.4f} ± "
        f"{sd_row['MLP TPR Drop']:.4f}"
    )

    print(
        f"MLP Evasion Rate : "
        f"{row['MLP Evasion Rate']:.4f} ± "
        f"{sd_row['MLP Evasion Rate']:.4f}"
    )

    print(
        f"MLP AUROC        : "
        f"{row['MLP AUROC']:.4f} ± "
        f"{sd_row['MLP AUROC']:.4f}"
    )

    print(
        f"MLP AUPRC        : "
        f"{row['MLP AUPRC']:.4f} ± "
        f"{sd_row['MLP AUPRC']:.4f}"
    )


# ============================================================
# 13. SAVE ALL FILES
# ============================================================

duration_by_seed.to_csv(
    "flow_duration_robustness_by_seed.csv",
    index=False
)

duration_summary.to_csv(
    "flow_duration_robustness_mean.csv",
    index=False
)

duration_sd.to_csv(
    "flow_duration_robustness_sd.csv",
    index=False
)


# ============================================================
# 14. COMBINED MEAN ± SD TABLE FOR PAPER
# ============================================================

paper_rows = []

for _, row in duration_summary.iterrows():

    scale = row["Duration Scale"]

    sd_row = duration_sd[
        duration_sd["Duration Scale"] == scale
    ].iloc[0]

    paper_rows.append({

        "Duration":
            f"{scale:.2f}x",

        "RF TPR":
            f"{row['RF TPR']:.4f} ± {sd_row['RF TPR']:.4f}",

        "RF TPR Drop":
            f"{row['RF TPR Drop']:.4f} ± "
            f"{sd_row['RF TPR Drop']:.4f}",

        "RF Evasion Rate":
            f"{row['RF Evasion Rate']:.4f} ± "
            f"{sd_row['RF Evasion Rate']:.4f}",

        "RF AUROC":
            f"{row['RF AUROC']:.4f} ± {sd_row['RF AUROC']:.4f}",

        "RF AUPRC":
            f"{row['RF AUPRC']:.4f} ± {sd_row['RF AUPRC']:.4f}",

        "MLP TPR":
            f"{row['MLP TPR']:.4f} ± {sd_row['MLP TPR']:.4f}",

        "MLP TPR Drop":
            f"{row['MLP TPR Drop']:.4f} ± "
            f"{sd_row['MLP TPR Drop']:.4f}",

        "MLP Evasion Rate":
            f"{row['MLP Evasion Rate']:.4f} ± "
            f"{sd_row['MLP Evasion Rate']:.4f}",

        "MLP AUROC":
            f"{row['MLP AUROC']:.4f} ± "
            f"{sd_row['MLP AUROC']:.4f}",

        "MLP AUPRC":
            f"{row['MLP AUPRC']:.4f} ± "
            f"{sd_row['MLP AUPRC']:.4f}"
    })


duration_paper_table = pd.DataFrame(
    paper_rows
)


duration_paper_table.to_csv(
    "flow_duration_robustness_paper_table.csv",
    index=False
)


# ============================================================
# 15. DISPLAY PAPER TABLE
# ============================================================

print("\n")
print("=" * 180)
print("PAPER TABLE — FLOW-DURATION ROBUSTNESS")
print("=" * 180)

print(
    duration_paper_table.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("FILES SAVED")
print("=" * 80)

print("1. flow_duration_robustness_by_seed.csv")
print("2. flow_duration_robustness_mean.csv")
print("3. flow_duration_robustness_sd.csv")
print("4. flow_duration_robustness_paper_table.csv")

EXPERIMENT A — FLOW-DURATION ROBUSTNESS
MLP device: cuda:0
Test samples: 82332
Duration feature: dur
Seeds: [42, 123, 2026]

CLEAN BASELINE TPR
------------------------------------------------------------
RF mean TPR  : 0.6829465572516839
MLP mean TPR : 0.423519809406159
Expected MLP : 0.423520


FLOW DURATION PERTURBATION = 0.95x

--------------------------------------------------------------------------------
SEED = 42
--------------------------------------------------------------------------------

RF
Clean TPR     : 0.716712
Perturbed TPR : 0.716514
TPR Drop      : 0.000199
Evasion Rate  : 0.283486
AUROC         : 0.805027
AUPRC         : 0.819746

MLP
Clean TPR     : 0.437483
Perturbed TPR : 0.437395
TPR Drop      : 0.000088
Evasion Rate  : 0.562605
AUROC         : 0.671880
AUPRC         : 0.753711

--------------------------------------------------------------------------------
SEED = 123
--------------------------------------------------------------------------------

RF
Clean T

In [26]:
# ============================================================
# EXPERIMENT B — TIMING / JITTER ROBUSTNESS
#
# Features:
#   sjit = source jitter
#   djit = destination jitter
#
# SAME FROZEN BASELINE MODELS
# NO RETRAINING
# NO PREPROCESSOR REFITTING
#
# VERIFIED CLEAN BASELINES:
#   RF  TPR = 0.682947
#   MLP TPR = 0.423520
#
# Metrics:
#   TPR
#   TPR Degradation
#   Evasion Rate
#   AUROC
#   AUPRC
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)


print("=" * 100)
print("EXPERIMENT B — TIMING / JITTER ROBUSTNESS")
print("=" * 100)


# ============================================================
# 1. CHECK REQUIRED VARIABLES
# ============================================================

assert "mlp_models" in globals(), "mlp_models not found"
assert "rf_models" in globals(), "rf_models not found"
assert "preprocessor" in globals(), "preprocessor not found"
assert "test_df" in globals(), "test_df not found"
assert "y_test" in globals(), "y_test not found"

assert "sjit" in test_df.columns, "sjit feature not found"
assert "djit" in test_df.columns, "djit feature not found"

SEEDS = [42, 123, 2026]


# ============================================================
# 2. VERIFIED CLEAN BASELINES
# ============================================================

RF_CLEAN_TPR = 0.682947
MLP_CLEAN_TPR = 0.423520

print("\nVERIFIED CLEAN BASELINES")
print("-" * 60)
print(f"RF  Clean TPR  = {RF_CLEAN_TPR:.6f}")
print(f"MLP Clean TPR  = {MLP_CLEAN_TPR:.6f}")


# ============================================================
# 3. PERTURBATION LEVELS
# ============================================================

# -10%, -5%, 0%, +5%, +10%

scales = [
    0.90,
    0.95,
    1.00,
    1.05,
    1.10
]


# ============================================================
# 4. DEVICE
# ============================================================

# Take device from one of the already-trained MLP models.
# No model is created or retrained here.

sample_mlp = mlp_models[SEEDS[0]]

device = next(
    sample_mlp.parameters()
).device

print(f"\nMLP device: {device}")


# ============================================================
# 5. PREPROCESSING
# ============================================================

def transform_test(df):

    X = preprocessor.transform(
        df.drop(
            columns=["label", "attack_cat"],
            errors="ignore"
        )
    )

    if hasattr(X, "toarray"):
        X = X.toarray()

    return np.asarray(
        X,
        dtype=np.float32
    )


# ============================================================
# 6. MLP PREDICTION
# ============================================================

def get_mlp_probability(mlp, X):

    mlp.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():

        logits = mlp(X_tensor)

        probabilities = torch.sigmoid(
            logits
        )

    return (
        probabilities
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


# ============================================================
# 7. RESULT STORAGE
# ============================================================

timing_results = []


# ============================================================
# 8. RUN TIMING / JITTER EXPERIMENT
# ============================================================

for feature in ["sjit", "djit"]:

    print("\n")
    print("=" * 100)
    print(f"TIMING FEATURE: {feature}")
    print("=" * 100)


    for scale in scales:

        print("\n" + "-" * 90)
        print(
            f"{feature} SCALE = {scale:.2f} "
            f"({(scale - 1) * 100:+.0f}%)"
        )
        print("-" * 90)


        # ====================================================
        # CREATE PERTURBED TEST SET
        # ====================================================

        transformed_df = test_df.copy()

        # ONLY selected timing feature is changed
        transformed_df[feature] = (
            transformed_df[feature]
            .astype(float)
            * scale
        )


        # ====================================================
        # PREPROCESS USING SAME FITTED PREPROCESSOR
        # ====================================================

        X_transformed = transform_test(
            transformed_df
        )


        # ====================================================
        # EACH FROZEN SEED
        # ====================================================

        for seed in SEEDS:

            print(f"\nSeed = {seed}")


            # =================================================
            # RANDOM FOREST
            # =================================================

            rf_model = rf_models[seed]

            rf_prob = rf_model.predict_proba(
                X_transformed
            )[:, 1]

            rf_pred = (
                rf_prob >= 0.5
            ).astype(int)


            # -------------------------------------------------
            # RF TPR
            # -------------------------------------------------

            rf_tpr = recall_score(
                y_test,
                rf_pred,
                zero_division=0
            )


            # -------------------------------------------------
            # RF TPR DEGRADATION
            #
            # Positive = degradation
            # Negative = improvement
            # -------------------------------------------------

            rf_tpr_degradation = (
                RF_CLEAN_TPR
                - rf_tpr
            )


            # -------------------------------------------------
            # RF EVASION RATE
            # -------------------------------------------------

            rf_evasion_rate = (
                1.0 - rf_tpr
            )


            # -------------------------------------------------
            # RF AUROC / AUPRC
            # -------------------------------------------------

            rf_auroc = roc_auc_score(
                y_test,
                rf_prob
            )

            rf_auprc = average_precision_score(
                y_test,
                rf_prob
            )


            # =================================================
            # MLP
            # =================================================

            mlp_model = mlp_models[seed]

            mlp_prob = get_mlp_probability(
                mlp_model,
                X_transformed
            )

            mlp_pred = (
                mlp_prob >= 0.5
            ).astype(int)


            # -------------------------------------------------
            # MLP TPR
            # -------------------------------------------------

            mlp_tpr = recall_score(
                y_test,
                mlp_pred,
                zero_division=0
            )


            # -------------------------------------------------
            # MLP TPR DEGRADATION
            #
            # Positive = degradation
            # Negative = improvement
            # -------------------------------------------------

            mlp_tpr_degradation = (
                MLP_CLEAN_TPR
                - mlp_tpr
            )


            # -------------------------------------------------
            # MLP EVASION RATE
            # -------------------------------------------------

            mlp_evasion_rate = (
                1.0 - mlp_tpr
            )


            # -------------------------------------------------
            # MLP AUROC / AUPRC
            # -------------------------------------------------

            mlp_auroc = roc_auc_score(
                y_test,
                mlp_prob
            )

            mlp_auprc = average_precision_score(
                y_test,
                mlp_prob
            )


            # =================================================
            # DISPLAY
            # =================================================

            print(
                f"RF  TPR              : {rf_tpr:.6f}"
            )

            print(
                f"RF  TPR Degradation  : "
                f"{rf_tpr_degradation:+.6f}"
            )

            print(
                f"RF  Evasion Rate     : "
                f"{rf_evasion_rate:.6f}"
            )

            print(
                f"RF  AUROC            : "
                f"{rf_auroc:.6f}"
            )

            print(
                f"RF  AUPRC            : "
                f"{rf_auprc:.6f}"
            )


            print()

            print(
                f"MLP TPR              : "
                f"{mlp_tpr:.6f}"
            )

            print(
                f"MLP TPR Degradation  : "
                f"{mlp_tpr_degradation:+.6f}"
            )

            print(
                f"MLP Evasion Rate     : "
                f"{mlp_evasion_rate:.6f}"
            )

            print(
                f"MLP AUROC            : "
                f"{mlp_auroc:.6f}"
            )

            print(
                f"MLP AUPRC            : "
                f"{mlp_auprc:.6f}"
            )


            # =================================================
            # SAVE
            # =================================================

            timing_results.append({

                "Feature": feature,

                "Scale": scale,

                "Perturbation_%": (
                    scale - 1
                ) * 100,

                "Seed": seed,


                # ---------------- RF ----------------

                "RF TPR": rf_tpr,

                "RF TPR Degradation":
                    rf_tpr_degradation,

                "RF Evasion Rate":
                    rf_evasion_rate,

                "RF AUROC":
                    rf_auroc,

                "RF AUPRC":
                    rf_auprc,


                # ---------------- MLP ----------------

                "MLP TPR": mlp_tpr,

                "MLP TPR Degradation":
                    mlp_tpr_degradation,

                "MLP Evasion Rate":
                    mlp_evasion_rate,

                "MLP AUROC":
                    mlp_auroc,

                "MLP AUPRC":
                    mlp_auprc
            })


# ============================================================
# 9. CONVERT TO DATAFRAME
# ============================================================

timing_by_seed = pd.DataFrame(
    timing_results
)


# ============================================================
# 10. MEAN ACROSS SEEDS
# ============================================================

timing_mean = (
    timing_by_seed
    .groupby(
        ["Feature", "Scale", "Perturbation_%"]
    )
    [
        [
            "RF TPR",
            "RF TPR Degradation",
            "RF Evasion Rate",
            "RF AUROC",
            "RF AUPRC",

            "MLP TPR",
            "MLP TPR Degradation",
            "MLP Evasion Rate",
            "MLP AUROC",
            "MLP AUPRC"
        ]
    ]
    .mean()
    .reset_index()
)


# ============================================================
# 11. STANDARD DEVIATION ACROSS SEEDS
# ============================================================

timing_sd = (
    timing_by_seed
    .groupby(
        ["Feature", "Scale", "Perturbation_%"]
    )
    [
        [
            "RF TPR",
            "RF TPR Degradation",
            "RF Evasion Rate",
            "RF AUROC",
            "RF AUPRC",

            "MLP TPR",
            "MLP TPR Degradation",
            "MLP Evasion Rate",
            "MLP AUROC",
            "MLP AUPRC"
        ]
    ]
    .std(ddof=1)
    .reset_index()
)


# ============================================================
# 12. FINAL MEAN ± SD TABLE
# ============================================================

print("\n\n")
print("=" * 150)
print("TIMING / JITTER ROBUSTNESS — MEAN ± SD")
print("=" * 150)


for _, row in timing_mean.iterrows():

    feature = row["Feature"]
    scale = row["Scale"]
    perturbation = row["Perturbation_%"]


    sd_row = timing_sd[
        (timing_sd["Feature"] == feature) &
        (timing_sd["Scale"] == scale)
    ].iloc[0]


    print("\n")
    print(
        f"{feature} | "
        f"{perturbation:+.0f}%"
    )

    print("-" * 90)


    # ========================================================
    # RF
    # ========================================================

    print(
        f"RF  TPR              : "
        f"{row['RF TPR']:.4f} ± "
        f"{sd_row['RF TPR']:.4f}"
    )

    print(
        f"RF  TPR Degradation  : "
        f"{row['RF TPR Degradation']:+.4f} ± "
        f"{sd_row['RF TPR Degradation']:.4f}"
    )

    print(
        f"RF  Evasion Rate     : "
        f"{row['RF Evasion Rate']:.4f} ± "
        f"{sd_row['RF Evasion Rate']:.4f}"
    )

    print(
        f"RF  AUROC            : "
        f"{row['RF AUROC']:.4f} ± "
        f"{sd_row['RF AUROC']:.4f}"
    )

    print(
        f"RF  AUPRC            : "
        f"{row['RF AUPRC']:.4f} ± "
        f"{sd_row['RF AUPRC']:.4f}"
    )


    print()


    # ========================================================
    # MLP
    # ========================================================

    print(
        f"MLP TPR              : "
        f"{row['MLP TPR']:.4f} ± "
        f"{sd_row['MLP TPR']:.4f}"
    )

    print(
        f"MLP TPR Degradation  : "
        f"{row['MLP TPR Degradation']:+.4f} ± "
        f"{sd_row['MLP TPR Degradation']:.4f}"
    )

    print(
        f"MLP Evasion Rate     : "
        f"{row['MLP Evasion Rate']:.4f} ± "
        f"{sd_row['MLP Evasion Rate']:.4f}"
    )

    print(
        f"MLP AUROC            : "
        f"{row['MLP AUROC']:.4f} ± "
        f"{sd_row['MLP AUROC']:.4f}"
    )

    print(
        f"MLP AUPRC            : "
        f"{row['MLP AUPRC']:.4f} ± "
        f"{sd_row['MLP AUPRC']:.4f}"
    )


# ============================================================
# 13. SAVE RESULTS
# ============================================================

timing_by_seed.to_csv(
    "timing_jitter_robustness_by_seed.csv",
    index=False
)

timing_mean.to_csv(
    "timing_jitter_robustness_mean.csv",
    index=False
)

timing_sd.to_csv(
    "timing_jitter_robustness_sd.csv",
    index=False
)


print("\n")
print("=" * 100)
print("FILES SAVED")
print("=" * 100)

print("timing_jitter_robustness_by_seed.csv")
print("timing_jitter_robustness_mean.csv")
print("timing_jitter_robustness_sd.csv")

EXPERIMENT B — TIMING / JITTER ROBUSTNESS

VERIFIED CLEAN BASELINES
------------------------------------------------------------
RF  Clean TPR  = 0.682947
MLP Clean TPR  = 0.423520

MLP device: cuda:0


TIMING FEATURE: sjit

------------------------------------------------------------------------------------------
sjit SCALE = 0.90 (-10%)
------------------------------------------------------------------------------------------

Seed = 42
RF  TPR              : 0.716624
RF  TPR Degradation  : -0.033677
RF  Evasion Rate     : 0.283376
RF  AUROC            : 0.805823
RF  AUPRC            : 0.819809

MLP TPR              : 0.437351
MLP TPR Degradation  : -0.013831
MLP Evasion Rate     : 0.562649
MLP AUROC            : 0.671983
MLP AUPRC            : 0.753905

Seed = 123
RF  TPR              : 0.657063
RF  TPR Degradation  : +0.025884
RF  Evasion Rate     : 0.342937
RF  AUROC            : 0.797332
RF  AUPRC            : 0.814463

MLP TPR              : 0.403666
MLP TPR Degradation  : +0.01

In [27]:
# ================================================================
# EXPERIMENT D — COMBINED DURATION + JITTER ROBUSTNESS
# ================================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)

print("=" * 100)
print("EXPERIMENT D — COMBINED DURATION + JITTER ROBUSTNESS")
print("=" * 100)


# ================================================================
# 1. CHECK REQUIRED VARIABLES
# ================================================================

required = [
    "test_df",
    "preprocessor",
    "rf_models",
    "mlp_models",
    "y_test"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}\n"
        "Run the clean baseline / model training cell first."
    )

SEEDS = [42, 123, 2026]

print("\nRequired variables found:")
print("  test_df")
print("  preprocessor")
print("  rf_models")
print("  mlp_models")
print("  y_test")

print("\nSeeds:", SEEDS)


# ================================================================
# 2. CHECK FEATURES
# ================================================================

TRANSFORM_FEATURES = [
    "dur",
    "sjit",
    "djit"
]

missing_features = [
    c for c in TRANSFORM_FEATURES
    if c not in test_df.columns
]

if missing_features:
    raise ValueError(
        f"Missing transformation features: {missing_features}"
    )

print("\nTest samples:", len(test_df))
print("Transformation features:", TRANSFORM_FEATURES)


# ================================================================
# 3. VERIFIED CLEAN BASELINES
# ================================================================

# These are the verified baselines from Experiment B
RF_CLEAN_TPR = 0.682947
MLP_CLEAN_TPR = 0.423520

print("\nVERIFIED CLEAN BASELINES")
print("-" * 70)
print(f"RF  Clean TPR  = {RF_CLEAN_TPR:.6f}")
print(f"MLP Clean TPR  = {MLP_CLEAN_TPR:.6f}")


# ================================================================
# 4. DEVICE
# ================================================================

sample_mlp = mlp_models[SEEDS[0]]

device = next(
    sample_mlp.parameters()
).device

print(f"\nMLP device: {device}")


# ================================================================
# 5. RAW TEST DATA
# ================================================================

DROP_COLUMNS = [
    "label",
    "attack_cat"
]

X_test_raw = test_df.drop(
    columns=[
        c for c in DROP_COLUMNS
        if c in test_df.columns
    ]
).copy()

y_test_arr = np.asarray(
    y_test
).astype(int)

print("Raw test shape:", X_test_raw.shape)
print("Positive samples:", np.sum(y_test_arr == 1))
print("Negative samples:", np.sum(y_test_arr == 0))


# ================================================================
# 6. PREPROCESSING FUNCTION
# ================================================================

def transform_test(df):

    X = preprocessor.transform(
        df.drop(
            columns=["label", "attack_cat"],
            errors="ignore"
        )
    )

    if hasattr(X, "toarray"):
        X = X.toarray()

    return np.asarray(
        X,
        dtype=np.float32
    )


# ================================================================
# 7. MLP PREDICTION FUNCTION
# ================================================================

def get_mlp_probability(mlp, X):

    mlp.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():

        logits = mlp(
            X_tensor
        ).view(-1)

        probabilities = torch.sigmoid(
            logits
        )

    return (
        probabilities
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


# ================================================================
# 8. RESULT STORAGE
# ================================================================

combined_results = []


# ================================================================
# 9. COMBINED TRANSFORMATIONS
# ================================================================

COMBINATIONS = [

    (
        "ALL CLEAN",
        1.00,
        1.00,
        1.00
    ),

    (
        "DUR -5% + SJIT -5% + DJIT -5%",
        0.95,
        0.95,
        0.95
    ),

    (
        "DUR +5% + SJIT +5% + DJIT +5%",
        1.05,
        1.05,
        1.05
    ),

    (
        "DUR -5% + SJIT +5% + DJIT +5%",
        0.95,
        1.05,
        1.05
    ),

    (
        "DUR +5% + SJIT -5% + DJIT -5%",
        1.05,
        0.95,
        0.95
    )
]


# ================================================================
# 10. RUN EXPERIMENT
# ================================================================

for name, dur_scale, sjit_scale, djit_scale in COMBINATIONS:

    print("\n")
    print("=" * 100)
    print(name)
    print("=" * 100)

    # ------------------------------------------------------------
    # CREATE TRANSFORMED TEST DATA
    # ------------------------------------------------------------

    transformed_df = X_test_raw.copy()

    # Duration
    transformed_df["dur"] = (
        pd.to_numeric(
            transformed_df["dur"],
            errors="coerce"
        ) * dur_scale
    )

    # Source jitter
    transformed_df["sjit"] = (
        pd.to_numeric(
            transformed_df["sjit"],
            errors="coerce"
        ) * sjit_scale
    )

    # Destination jitter
    transformed_df["djit"] = (
        pd.to_numeric(
            transformed_df["djit"],
            errors="coerce"
        ) * djit_scale
    )


    # ------------------------------------------------------------
    # SAME FITTED PREPROCESSOR
    # ------------------------------------------------------------

    X_transformed = transform_test(
        transformed_df
    )


    # ------------------------------------------------------------
    # EACH FROZEN SEED
    # ------------------------------------------------------------

    for seed in SEEDS:

        print(f"\nSeed = {seed}")

        # ========================================================
        # RANDOM FOREST
        # ========================================================

        rf_model = rf_models[seed]

        rf_prob = rf_model.predict_proba(
            X_transformed
        )[:, 1]

        rf_pred = (
            rf_prob >= 0.5
        ).astype(int)

        rf_tpr = recall_score(
            y_test_arr,
            rf_pred,
            zero_division=0
        )

        rf_tpr_degradation = (
            RF_CLEAN_TPR - rf_tpr
        )

        rf_evasion_rate = (
            1.0 - rf_tpr
        )

        rf_auroc = roc_auc_score(
            y_test_arr,
            rf_prob
        )

        rf_auprc = average_precision_score(
            y_test_arr,
            rf_prob
        )


        # ========================================================
        # MLP
        # ========================================================

        mlp_model = mlp_models[seed]

        mlp_prob = get_mlp_probability(
            mlp_model,
            X_transformed
        )

        mlp_pred = (
            mlp_prob >= 0.5
        ).astype(int)

        mlp_tpr = recall_score(
            y_test_arr,
            mlp_pred,
            zero_division=0
        )

        mlp_tpr_degradation = (
            MLP_CLEAN_TPR - mlp_tpr
        )

        mlp_evasion_rate = (
            1.0 - mlp_tpr
        )

        mlp_auroc = roc_auc_score(
            y_test_arr,
            mlp_prob
        )

        mlp_auprc = average_precision_score(
            y_test_arr,
            mlp_prob
        )


        # ========================================================
        # DISPLAY
        # ========================================================

        print(
            f"RF  TPR              : {rf_tpr:.6f}"
        )

        print(
            f"RF  TPR Degradation  : "
            f"{rf_tpr_degradation:+.6f}"
        )

        print(
            f"RF  Evasion Rate     : "
            f"{rf_evasion_rate:.6f}"
        )

        print(
            f"RF  AUROC            : "
            f"{rf_auroc:.6f}"
        )

        print(
            f"RF  AUPRC            : "
            f"{rf_auprc:.6f}"
        )

        print()

        print(
            f"MLP TPR              : "
            f"{mlp_tpr:.6f}"
        )

        print(
            f"MLP TPR Degradation  : "
            f"{mlp_tpr_degradation:+.6f}"
        )

        print(
            f"MLP Evasion Rate     : "
            f"{mlp_evasion_rate:.6f}"
        )

        print(
            f"MLP AUROC            : "
            f"{mlp_auroc:.6f}"
        )

        print(
            f"MLP AUPRC            : "
            f"{mlp_auprc:.6f}"
        )


        # ========================================================
        # SAVE SEED RESULT
        # ========================================================

        combined_results.append({

            "Transformation": name,

            "DUR Scale": dur_scale,
            "SJIT Scale": sjit_scale,
            "DJIT Scale": djit_scale,

            "Seed": seed,

            # ---------------- RF ----------------

            "RF TPR": rf_tpr,

            "RF TPR Degradation":
                rf_tpr_degradation,

            "RF Evasion Rate":
                rf_evasion_rate,

            "RF AUROC":
                rf_auroc,

            "RF AUPRC":
                rf_auprc,

            # ---------------- MLP ----------------

            "MLP TPR": mlp_tpr,

            "MLP TPR Degradation":
                mlp_tpr_degradation,

            "MLP Evasion Rate":
                mlp_evasion_rate,

            "MLP AUROC":
                mlp_auroc,

            "MLP AUPRC":
                mlp_auprc
        })


# ================================================================
# 11. BY-SEED DATAFRAME
# ================================================================

combined_by_seed = pd.DataFrame(
    combined_results
)


# ================================================================
# 12. MEAN ACROSS SEEDS
# ================================================================

METRIC_COLUMNS = [

    "RF TPR",
    "RF TPR Degradation",
    "RF Evasion Rate",
    "RF AUROC",
    "RF AUPRC",

    "MLP TPR",
    "MLP TPR Degradation",
    "MLP Evasion Rate",
    "MLP AUROC",
    "MLP AUPRC"
]

combined_mean = (
    combined_by_seed
    .groupby(
        [
            "Transformation",
            "DUR Scale",
            "SJIT Scale",
            "DJIT Scale"
        ]
    )[METRIC_COLUMNS]
    .mean()
    .reset_index()
)


# ================================================================
# 13. STANDARD DEVIATION ACROSS SEEDS
# ================================================================

combined_sd = (
    combined_by_seed
    .groupby(
        [
            "Transformation",
            "DUR Scale",
            "SJIT Scale",
            "DJIT Scale"
        ]
    )[METRIC_COLUMNS]
    .std(ddof=1)
    .reset_index()
)


# ================================================================
# 14. FINAL MEAN ± SD TABLE
# ================================================================

print("\n\n")
print("=" * 150)
print("COMBINED DURATION + JITTER ROBUSTNESS — MEAN ± SD")
print("=" * 150)


for _, row in combined_mean.iterrows():

    transformation = row["Transformation"]

    sd_row = combined_sd[
        combined_sd["Transformation"] == transformation
    ].iloc[0]

    print("\n")
    print(transformation)
    print("-" * 100)

    print(
        f"RF  TPR              : "
        f"{row['RF TPR']:.4f} ± "
        f"{sd_row['RF TPR']:.4f}"
    )

    print(
        f"RF  TPR Degradation  : "
        f"{row['RF TPR Degradation']:+.4f} ± "
        f"{sd_row['RF TPR Degradation']:.4f}"
    )

    print(
        f"RF  Evasion Rate     : "
        f"{row['RF Evasion Rate']:.4f} ± "
        f"{sd_row['RF Evasion Rate']:.4f}"
    )

    print(
        f"RF  AUROC            : "
        f"{row['RF AUROC']:.4f} ± "
        f"{sd_row['RF AUROC']:.4f}"
    )

    print(
        f"RF  AUPRC            : "
        f"{row['RF AUPRC']:.4f} ± "
        f"{sd_row['RF AUPRC']:.4f}"
    )

    print()

    print(
        f"MLP TPR              : "
        f"{row['MLP TPR']:.4f} ± "
        f"{sd_row['MLP TPR']:.4f}"
    )

    print(
        f"MLP TPR Degradation  : "
        f"{row['MLP TPR Degradation']:+.4f} ± "
        f"{sd_row['MLP TPR Degradation']:.4f}"
    )

    print(
        f"MLP Evasion Rate     : "
        f"{row['MLP Evasion Rate']:.4f} ± "
        f"{sd_row['MLP Evasion Rate']:.4f}"
    )

    print(
        f"MLP AUROC            : "
        f"{row['MLP AUROC']:.4f} ± "
        f"{sd_row['MLP AUROC']:.4f}"
    )

    print(
        f"MLP AUPRC            : "
        f"{row['MLP AUPRC']:.4f} ± "
        f"{sd_row['MLP AUPRC']:.4f}"
    )


# ================================================================
# 15. SAVE RESULTS
# ================================================================

combined_by_seed.to_csv(
    "combined_duration_jitter_robustness_by_seed.csv",
    index=False
)

combined_mean.to_csv(
    "combined_duration_jitter_robustness_mean.csv",
    index=False
)

combined_sd.to_csv(
    "combined_duration_jitter_robustness_sd.csv",
    index=False
)


# ================================================================
# 16. FINAL OUTPUT
# ================================================================

print("\n")
print("=" * 100)
print("FILES SAVED")
print("=" * 100)

print(
    "combined_duration_jitter_robustness_by_seed.csv"
)

print(
    "combined_duration_jitter_robustness_mean.csv"
)

print(
    "combined_duration_jitter_robustness_sd.csv"
)

EXPERIMENT D — COMBINED DURATION + JITTER ROBUSTNESS

Required variables found:
  test_df
  preprocessor
  rf_models
  mlp_models
  y_test

Seeds: [42, 123, 2026]

Test samples: 82332
Transformation features: ['dur', 'sjit', 'djit']

VERIFIED CLEAN BASELINES
----------------------------------------------------------------------
RF  Clean TPR  = 0.682947
MLP Clean TPR  = 0.423520

MLP device: cuda:0
Raw test shape: (82332, 43)
Positive samples: 45332
Negative samples: 37000


ALL CLEAN

Seed = 42
RF  TPR              : 0.716712
RF  TPR Degradation  : -0.033765
RF  Evasion Rate     : 0.283288
RF  AUROC            : 0.805111
RF  AUPRC            : 0.819160

MLP TPR              : 0.437483
MLP TPR Degradation  : -0.013963
MLP Evasion Rate     : 0.562517
MLP AUROC            : 0.671884
MLP AUPRC            : 0.753734

Seed = 123
RF  TPR              : 0.656755
RF  TPR Degradation  : +0.026192
RF  Evasion Rate     : 0.343245
RF  AUROC            : 0.796718
RF  AUPRC            : 0.813893

ML